<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/particle_bouncing_inside_circle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Parameters
RADIUS = 10
NUM_BOUNCES = 1000
FPS = 30

def generate_ray_path(n_points, radius):
    points = [np.array([radius, 0.0])]
    angle = np.pi / 4  # Initial direction

    for _ in range(n_points):
        p = points[-1]
        # Find intersection with circle: |p + t*v|^2 = R^2
        v = np.array([np.cos(angle), np.sin(angle)])

        # Quadratic: t^2|v|^2 + 2t(p.v) + |p|^2 - R^2 = 0
        # Since |v|=1: t^2 + 2t(p.v) + (|p|^2 - R^2) = 0
        b = 2 * np.dot(p, v)
        c = np.dot(p, p) - radius**2
        delta = b**2 - 4*c

        if delta < 0: break

        # We want the positive root (forward in time)
        t = (-b + np.sqrt(delta)) / 2
        if t < 1e-6: # If at boundary, find the other root
            t = (-b + np.sqrt(delta)) / 2

        next_p = p + t * v
        points.append(next_p)
        # Turn 90 degrees inward
        angle += np.pi / 2

    return np.array(points)

# Generate high-resolution path
path = generate_ray_path(NUM_BOUNCES, RADIUS)

# Animation Setup
fig, ax = plt.subplots(figsize=(8, 8), facecolor='black')
ax.set_xlim(-RADIUS*1.2, RADIUS*1.2)
ax.set_ylim(-RADIUS*1.2, RADIUS*1.2)
ax.set_axis_off()

# Drawing the boundary
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(RADIUS*np.cos(theta), RADIUS*np.sin(theta), color='white', alpha=0.2, lw=1)

line, = ax.plot([], [], lw=0.5, alpha=0.7, color='#00FFCC')

def update(frame):
    # Simulate 3D rotation by projecting the 2D path
    rot_angle = frame * 0.02
    cos_a, sin_a = np.cos(rot_angle), np.sin(rot_angle)

    # Dynamic slice of the path to show growth
    current_len = min(frame * 5, len(path))
    pts = path[:current_len]

    # Apply rotation/perspective projection
    x_rot = pts[:, 0] * cos_a
    y_rot = pts[:, 1]
    z_depth = pts[:, 0] * sin_a

    # Simple perspective factor
    factor = 1 / (1.5 + z_depth / RADIUS)
    line.set_data(x_rot * factor, y_rot * factor)

    # Color shift based on frame
    line.set_color(plt.cm.magma(frame / 200))
    return line,

ani = FuncAnimation(fig, update, frames=200, interval=50, blit=True)
plt.close()
HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
from matplotlib.collections import LineCollection
from IPython.display import HTML

# ── Adjustable Parameters ─────────────────────────────────────────────────────
TURN_DEG     = 80.54          # ADJUST THIS: turn angle in degrees
START_DIST   = 0.80           # ADJUST THIS: starting distance fraction (0.0 to 1.0)

RADIUS       = 1.0            # circle radius
MAX_CHORDS   = 2000           # total chords to draw
CHORDS_PER_FRAME = 15         # animation speed
COLORMAP     = "plasma"       # try: inferno, viridis, cool, twilight, hsv
FIGSIZE      = (8, 8)
# ──────────────────────────────────────────────────────────────────────────────

TURN_RAD = np.radians(TURN_DEG)
R = RADIUS

def rotate(dx, dy, angle):
    c, s = np.cos(angle), np.sin(angle)
    return c * dx - s * dy, s * dx + c * dy

def ray_circle_t(px, py, dx, dy, r=R):
    b = 2 * (px * dx + py * dy)
    c = px * px + py * py - r * r
    disc = b * b - 4 * c
    if disc < 0: return None
    sq = np.sqrt(disc)
    t1, t2 = (-b - sq) / 2, (-b + sq) / 2
    candidates = [t for t in (t1, t2) if t > 1e-9]
    return min(candidates) if candidates else None

def build_chords(turn_rad, start_frac, seed_angle, max_chords):
    sr = start_frac * R * 0.98
    lx, ly = sr * np.cos(seed_angle), sr * np.sin(seed_angle)
    init_angle = seed_angle + np.pi * 0.37
    dx, dy = np.cos(init_angle), np.sin(init_angle)

    chords = []
    for _ in range(max_chords):
        t = ray_circle_t(lx, ly, dx, dy)
        if t is None: break
        bx, by = lx + dx * t, ly + dy * t
        chords.append(((lx, ly), (bx, by)))
        nx, ny = bx / R, by / R
        rCW  = rotate(dx, dy, -turn_rad)
        rCCW = rotate(dx, dy,  turn_rad)
        dot_cw  = rCW[0] * nx + rCW[1] * ny
        dot_ccw = rCCW[0] * nx + rCCW[1] * ny
        chosen = rCW if dot_cw < 0 else rCCW
        lx, ly, (dx, dy) = bx, by, chosen
    return chords

def make_segments_and_colors(chords, cmap_name):
    n = len(chords)
    segments = [[(x1, y1), (x2, y2)] for (x1, y1), (x2, y2) in chords]
    cmap = plt.get_cmap(cmap_name)
    colors = []
    for i in range(n):
        t = (i % 500) / 500
        alpha = 0.2 + 0.6 * np.abs(np.sin(i * 0.05))
        rgba = list(cmap(t))
        rgba[3] = alpha
        colors.append(rgba)
    linewidths = [0.5 + 0.5 * np.abs(np.sin(i * 0.05)) for i in range(n)]
    return segments, colors, linewidths

# Build
seed_angle = np.random.uniform(0, 2 * np.pi)
chords = build_chords(TURN_RAD, START_DIST, seed_angle, MAX_CHORDS)
segments, colors, linewidths = make_segments_and_colors(chords, COLORMAP)

# Figure setup - Fully transparent background
fig, ax = plt.subplots(figsize=FIGSIZE)
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)
ax.set_aspect("equal")
ax.set_xlim(-R * 1.1, R * 1.1)
ax.set_ylim(-R * 1.1, R * 1.1)
ax.axis("off")

# Invisible/Transparent boundary
rim = Circle((0, 0), R, color='white', alpha=0.1, fill=False, lw=1)
ax.add_patch(rim)

lc = LineCollection([], colors=[], linewidths=[], capstyle="round")
ax.add_collection(lc)
particle, = ax.plot([], [], "o", color="white", ms=3, alpha=0.8)

def update(frame):
    end = min(frame * CHORDS_PER_FRAME, len(chords))
    if end > 0:
        lc.set_segments(segments[:end])
        lc.set_colors(colors[:end])
        lc.set_linewidths(linewidths[:end])
        ex, ey = chords[end - 1][1]
        particle.set_data([ex], [ey])
    return lc, particle

ani = animation.FuncAnimation(
    fig, update, frames=len(chords) // CHORDS_PER_FRAME + 5,
    interval=30, blit=True
)

plt.close()
HTML(ani.to_jshtml())

Buffered data was truncated after reaching the output size limit.